In [ ]:
%load_ext autoreload
%autoreload 2
import os
if not hasattr(__builtins__, '_cwd_set'):
    os.chdir('..')
    __builtins__._cwd_set = True

In [ ]:
from pathlib import Path
from pprint import pprint

import httpx

In [ ]:
from util import util

In [ ]:
MODEL_PATH = Path('data/models/setValueTest.xml')
source_xml = MODEL_PATH.read_text(encoding='utf-8')
source_graph = util.import_xml(source_xml)
source_marking = util.marking(source_graph)

exported_xml = util.export_xml(source_graph)
local_roundtrip_graph = util.import_xml(exported_xml)
local_roundtrip_marking = util.marking(local_roundtrip_graph)

print('Generated XML characters:', len(exported_xml))
print('Marking difference after local round trip:')
pprint(util.marking_difference(source_marking, local_roundtrip_marking))

In [ ]:
source_graph

In [ ]:
API_URL = 'http://127.0.0.1:8000/api/chat/response'
DCR_CHAT = 1
request_body = {
    'text': 'Starting DCR Chat session',
    'chat_type': DCR_CHAT,
    'graph_xml': exported_xml,  # Complete XML string in the JSON body.
    'dcr_role': 'Citizen'
}
print({**request_body, 'graph_xml': f'<{len(exported_xml)} XML characters>'})
execution_result = httpx.post(API_URL, json=request_body, timeout=30).json()

In [ ]:
session_id = execution_result['session_id']
dcr_graph = util.import_xml(execution_result['graph_xml'])
backend_marking = util.marking(dcr_graph)

In [ ]:
HISTORY_URL = 'http://127.0.0.1:8000/api/chat/history'
hist_response = httpx.post(HISTORY_URL, json={"session_id":session_id}, timeout=30)
hist_response.json()

In [ ]:
print(execution_result['text'])

In [ ]:
answer = "I was born in 2015"
request_body = {
    'text': answer,
    'session_id': session_id,
    'act_id': execution_result['act_id'],
    'dcr_role': 'Citizen'
}
execution_result = httpx.post(API_URL, json=request_body, timeout=30).json()

In [ ]:
from pm4py.objects.dcr.ocdcr import obj
ed1 = obj.DcrEventData(name="age", data_type=int)
act1 = obj.DcrActivity("Event_act1",label="Age",role="Citizen", takesInput=True, eventData=ed1,priority=1)
ed2 = obj.DcrEventData(name="threshold", data_type=bool)
act2 = obj.DcrActivity("Event_act2",label="Age Threshold",role="Robot", takesInput=True, eventData=ed2,priority=2)
relations = {
    obj.DcrSetValue(act1, act2, [("source", "data"), ">=", 18])
}
graph = obj.DcrGraph("testGraph", elements={act1, act2},relations=relations)
from pm4py.objects.dcr.exporter import exporter as dcr_exporter
dcr_exporter.apply(graph,'/home/vco/Projects2026/DcrController/backend/data/models/setValueTestExport.xml',dcr_exporter.Variants.DCR_JS_PORTAL)
from pm4py.objects.dcr.ocdcr.semantics import DcrSemantics

dcr_semantics = DcrSemantics()
dcr_semantics.executeActivity(obj.DcrExecution("Event_act1", 18), graph)
dcr_semantics.executeActivity(obj.DcrExecution("Event_act2"), graph)
for e in graph.elements:
    print(e.data)

In [ ]:
dcr_semantics = DcrSemantics()
dcr_semantics.executeActivity(obj.DcrExecution("Event_act1", 18), source_graph)
dcr_semantics.executeActivity(obj.DcrExecution("Event_act2"), source_graph)
for e in graph.elements:
    print(e.data)

In [ ]:
enabled_events = set()
enabled_pending = set()
for element in dcr_graph.elements:
    if dcr_semantics.isEnabled(element,dcr_graph):
        enabled_events.add(element)
        if element.pending:
            enabled_pending.add(element)
activity = None
for e in enabled_pending:
    print(e.ID, e.label, e.computation)
    if e.ID == execution_result['act_id']:
        activity = e

In [ ]:
activity.eventData.data_type

In [ ]:
from tools.interpret_input import InterpretInput

In [ ]:
ii_llm_tool = InterpretInput()

In [ ]:
res = await ii_llm_tool.get_closest_match(execution_body['text'],activity.eventData.data_type)

In [ ]:
res

In [ ]:
from pm4py.objects.dcr.ocdcr.obj import DcrExecution


execution = DcrExecution(execution_body['act_id'], 
                         role=execution_body['dcr_role'],
                         input=res)

In [ ]:
dcr_semantics.executeActivity(execution,dcr_graph)

In [ ]:
execution_result

In [ ]:

updated_graph = util.import_xml(execution_result['graph_xml'])
updated_marking = util.marking(updated_graph)

print('Marking difference after backend execution and XML round trip:')
pprint(util.marking_difference(backend_marking, updated_marking))
print('Next enabled activity:', execution_result['act_id'], execution_result['text'])

In [ ]:
from util.csvparser import CsvParser
from util.fileprocessor import FileProcessor
from util.jsonparser import JsonParser
from util.pdfparser import LocalPdfParser
from util.textparser import TextParser
from util.textsplitter import SentenceTextSplitter, SimpleTextSplitter, XmlSplitter

csv_max_chars_per_page = 1000
sentence_text_splitter = SentenceTextSplitter()
file_processors = {
    ".json": FileProcessor(JsonParser(), SimpleTextSplitter()),
    ".xml": FileProcessor(TextParser(), XmlSplitter()),
    ".md": FileProcessor(TextParser(), sentence_text_splitter),
    ".txt": FileProcessor(TextParser(), sentence_text_splitter),
    ".csv": FileProcessor(CsvParser(max_chars_per_page=csv_max_chars_per_page), sentence_text_splitter),
    ".pdf": FileProcessor(LocalPdfParser(), sentence_text_splitter),
}

In [ ]:
fp = FileProcessor(LocalPdfParser(), sentence_text_splitter)

In [ ]:
fp.parser.parse()

In [ ]:
from tools.find_relevant_dcr_graphs import FindRelevantDcrGraphs

query="Lexplain article 86"
dcr_finder = FindRelevantDcrGraphs()
for res in dcr_finder.find(query=query, top_k=2):
    print(res.score,res.format,res.source)

In [ ]:
from tools.find_similar_cases import FindSimilarCases

finder = FindSimilarCases()

results = finder.find("child disability expenses", top_k=5)
clusters = finder.cluster("child disability expenses", top_k_per_outcome=5)

print(clusters.positive)
print(clusters.negative)
print(clusters.unknown)
for result in results:
    print(result.score, result.source, result.page_number)#, result.text)

In [ ]:
from tools.find_relevant_laws import FindRelevantLaws

results = FindRelevantLaws().find("requirements for compensation", top_k=5)

for result in results:
    print(result.score, result.source, result.page_number)#, result.text)

In [ ]:
delete_response = httpx.request(
    'DELETE',
    'http://127.0.0.1:8000/api/chat/session',
    json={'session_id': execution_result['session_id']},
)
delete_response.raise_for_status()
print('Session removed from backend memory.')

In [ ]:
from sentence_transformers import SentenceTransformer

# Download from the 🤗 Hub
model = SentenceTransformer("google/embeddinggemma-300m")

# 2. Save to a local directory
model.save_pretrained("models/local_gemma_embedding")
print("Model successfully saved locally!")

In [ ]:
model = SentenceTransformer("models/local_gemma_embedding")
# Run inference with queries and documents
query = "Which planet is known as the Red Planet?"
documents = [
    "Venus is often called Earth's twin because of its similar size and proximity.",
    "Mars, known for its reddish appearance, is often referred to as the Red Planet.",
    "Jupiter, the largest planet in our solar system, has a prominent red spot.",
    "Saturn, famous for its rings, is sometimes mistaken for the Red Planet."
]
query_embeddings = model.encode_query(query)
document_embeddings = model.encode_document(documents)
print(query_embeddings.shape, document_embeddings.shape)
# (768,) (4, 768)

# Compute similarities to determine a ranking
similarities = model.similarity(query_embeddings, document_embeddings)
print(similarities)
# tensor([[0.3011, 0.6359, 0.4930, 0.4889]])
